In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple
import math

import torch
import torch.nn.functional as F
import torch.nn as nn

!pip install gluonts numpy

In [ ]:
from gluonts.dataset.repository.datasets import get_dataset

# Load Tourism Monthly dataset via GluonTS
dataset = get_dataset("tourism_monthly")

# Extract time series as numpy arrays
all_series = [np.array(entry["target"], dtype=np.float32) for entry in dataset.train]

# Get dataset statistics
lengths = [len(s) for s in all_series]
print(f"Number of series: {len(all_series)}")
print(f"Length range: {min(lengths)} - {max(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f}")
print(f"Total data points: {sum(lengths):,}")

In [ ]:
# Visualize sample series
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
axes = axes.flatten()

sample_indices = [0, 50, 150]

for ax, idx in zip(axes, sample_indices):
    if idx < len(all_series):
        series = all_series[idx]
        ax.plot(series, linewidth=1.5, color='steelblue')
        ax.set_title(f'Series {idx} (n={len(series)})', fontsize=10)
        ax.set_xlabel('Time (months)')
        ax.set_ylabel('Value')

plt.suptitle('Sample Time Series from Tourism Monthly Dataset', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Select context/prediction lengths
series = next(series for series in all_series if len(series) > 150)
context_length = 60
prediction_length = 24
assert len(series) > context_length + prediction_length, "Series too short for chosen context/prediction lengths"
context = series[-(context_length + prediction_length):-prediction_length]
future = series[-prediction_length:]  # For comparison
print(f"Context shape: {context.shape}, Future shape: {future.shape}")

In [ ]:
!pip install timesfm

In [ ]:
from timesfm import TimesFM_2p5_200M_torch

# Prepare input for TimesFM
input_series = context.astype('float32')

# Load the PyTorch-based TimesFM 2.5 model directly
# The config is typically handled within the model or passed during the forecast call
model = TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

In [ ]:
# Run TimesFM forecast (point + quantiles) using the 2.5 PyTorch API
import torch
import timesfm

# Prepare input as a tensor [Batch, Context_Length]
forecast_input_tensor = torch.tensor([input_series], dtype=torch.float32)

# In TimesFM 2.5, we use ForecastConfig to define the inference constraints
config = timesfm.ForecastConfig(
    max_context=128,
    max_horizon=prediction_length,
    normalize_inputs=True
)

# Compile the model with the forecast configuration
model.compile(forecast_config=config)

# Perform forecast with the required 'inputs' and 'horizon' arguments
timesfm_point, timesfm_quantiles = model.forecast(
    inputs=forecast_input_tensor,
    horizon=prediction_length
)

# Extract results for plotting
# TimesFM 2.5 returns numpy arrays directly from the forecast method
point_forecast = timesfm_point[0]
quantiles = timesfm_quantiles[0]

print(f"Point forecast shape: {point_forecast.shape}")
print(f"Quantiles shape: {quantiles.shape}")

In [ ]:
# Plot context, ground truth, and TimesFM forecast with quantile bands
plt.figure(figsize=(12, 5))
x_context = np.arange(len(context))
x_future = np.arange(len(context), len(context) + len(future))
x_forecast = np.arange(len(context), len(context) + len(point_forecast))

plt.plot(x_context, context, label="Context (history)", color="steelblue")
plt.plot(x_future, future, label="Ground Truth (future)", color="black", linestyle=":")
plt.plot(x_forecast, point_forecast, label="TimesFM Point Forecast", color="crimson")

# Plot 80% (from 10 to 90%) and 90% quantile bands
plt.fill_between(x_forecast, quantiles[:, 0], quantiles[:, 8], color="crimson", alpha=0.2, label="80% CI")
plt.fill_between(x_forecast, quantiles[:, 0], quantiles[:, 9], color="crimson", alpha=0.1, label="~90% CI")

plt.axvline(len(context)-0.5, color="gray", linestyle="--", label="Forecast Start")
plt.legend()
plt.title("TimesFM Forecast: Tourism Monthly")
plt.xlabel("Time (months)")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

In [ ]:
# Forecast Evaluation Metrics (reusable helper)

def compute_metrics(pred, actual):
    """
    Compute MAE, RMSE, MAPE for forecast evaluation.
    Works with both PyTorch tensors and numpy arrays.
    """
    if isinstance(pred, torch.Tensor):
        mae = F.l1_loss(pred, actual).item()
        rmse = torch.sqrt(F.mse_loss(pred, actual)).item()
        mape = (torch.mean(torch.abs((pred - actual) / (torch.abs(actual) + 1e-8))) * 100).item()
    else:
        pred, actual = np.array(pred), np.array(actual)
        mae = np.mean(np.abs(pred - actual))
        rmse = np.sqrt(np.mean((pred - actual)**2))
        mape = np.mean(np.abs((pred - actual) / (np.abs(actual) + 1e-8))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

metrics = compute_metrics(point_forecast, future)

print(f"MAE: {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"MAPE: {metrics['MAPE']:.2f}%")

In [ ]:
# Patching: simple demonstration of a time series into fixed-length patches

import numpy as np
import matplotlib.pyplot as plt

# Example time series (64 time steps breaking into 8 patches of length 8)
time_series = np.arange(1, 65)
print(f"Original time series shape: {time_series.shape}")
print(f"Original time series: {time_series}")


# Split into patches
patch_len = 8
num_patches = len(time_series) // patch_len
patches = time_series[:num_patches * patch_len].reshape(num_patches, patch_len)
print(f"Patches shape: {patches.shape}")
print(f"Patches: {patches}")

# Visualize patches
fig, ax = plt.subplots(figsize=(10, 2))
for i, patch in enumerate(patches):
    ax.plot(range(i * patch_len, (i + 1) * patch_len), patch, marker='o', label=f'Patch {i+1}')
ax.set_title(f'Time Series Split into Patches (patch_len={patch_len})')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# From paper: The job of the input layers is to preprocess the time-series into input tokens to the transformer layers.

def _masked_mean_std(inputs, padding, min_valid=3, tolerance=1e-6):
    """
    Compute mean and std from the first patch with enough valid (non-padded) values.

    TimesFM normalizes each time series by statistics computed from a single "reference"
    patch - the first patch that has at least `min_valid` non-padded values. This ensures
    consistent normalization even when early patches are heavily padded.
    """
    # Find patches with at least min_valid non-pad values
    valid_counts = (padding == 0).sum(dim=2)  # (B, N)
    # For each batch, get index of first patch with >= min_valid valid values
    def get_patch_index(arr):
        indices = torch.argmax((arr >= min_valid).to(torch.int32), dim=1)
        row_sum = (arr >= min_valid).to(torch.int32).sum(dim=1)
        return torch.where(row_sum == 0, arr.shape[1] - 1, indices)
    patch_indices = get_patch_index(valid_counts)  # (B,)
    bidxs = torch.arange(inputs.shape[0])
    arr = inputs[bidxs, patch_indices, :]  # (B, P)
    pad = padding[bidxs, patch_indices, :]  # (B, P)
    mask = 1 - pad  # (B, P)
    num_valid = torch.sum(mask, dim=1).clamp(min=1.0)
    masked_sum = torch.sum(arr * mask, dim=1)
    masked_mean = masked_sum / num_valid
    centered = (arr - masked_mean.unsqueeze(-1)) * mask
    masked_var = torch.sum(centered ** 2, dim=1) / num_valid
    masked_var = torch.clamp(masked_var, min=0.0)
    masked_std = torch.sqrt(masked_var).clamp(min=tolerance)
    return masked_mean, masked_std


def _forward_transform(patches, pad_mask=None, pad_val=0.0, tolerance=1e-6):
    """
    Normalize patches using mean/std from a reference patch.

    This is TimesFM's input normalization: z-score normalize the entire time series
    using statistics from a single reference patch (first patch with enough valid values).
    Padded positions are set to a constant value after normalization.
    """
    if pad_mask is None:
        pad_mask = torch.zeros_like(patches)
    mu, sigma = _masked_mean_std(patches, pad_mask, tolerance=tolerance)
    # Normalize all patches
    # Reshape mu/sigma from [B] to [B, 1, 1] for broadcasting over [B, N, P]
    norm_patches = (patches - mu.view(-1, 1, 1)) / sigma.view(-1, 1, 1)
    # Set pad values back to pad_val
    norm_patches = torch.where(torch.abs(pad_mask - 1.0) < tolerance,
                              torch.tensor(pad_val, dtype=norm_patches.dtype, device=norm_patches.device),
                              norm_patches)
    return norm_patches, (mu, sigma)

In [ ]:
# Example usage for new _forward_transform() (with batch dimension)
patches_torch = torch.tensor(patches.astype(np.float32))  # (num_patches, patch_len)
# Add batch dimension: (B, N, P)
patches_batched = patches_torch.unsqueeze(0)  # (1, num_patches, patch_len)
# Create a pad_mask that masks some values (e.g., mask last 2 values in each patch)
pad_mask = torch.zeros_like(patches_batched)
pad_mask[:, :, -2:] = 1  # mask last 2 values in every patch
pad_mask[:, 0, :] = 1    # mask entire first patch to test fallback
norm_patches, (mu, sigma) = _forward_transform(patches_batched, pad_mask)
print('Original patches shape:', patches_batched.shape)
print('Normalized patches shape:', norm_patches.shape)
print('mu:', mu.detach().cpu().numpy(), 'sigma:', sigma.detach().cpu().numpy())

# Show that mean and std are the same as for the first row with >=3 valid values (row 1)
row_idx = 1
row_vals = patches_batched[0, row_idx, :][pad_mask[0, row_idx, :] == 0]
row_mean = row_vals.mean().item()
row_std = row_vals.std(unbiased=False).item()
print(f"Row {row_idx} valid values: {row_vals}")
print(f"Row {row_idx} mean: {row_mean}, std: {row_std}")
print(f"Row {row_idx} normalized values:\n{norm_patches[0, row_idx, :].detach().cpu().numpy()}")

In [ ]:
# From paper: Then each patch is processed by a Residual Block
# into a vector of size model_dim. The Residual Block is essentially a Multi-layer
# Perceptron (MLP) block with one hidden layer with a skip connection.

class ResidualBlock(nn.Module):
    """
    Residual MLP block for projecting patches to embeddings.

    This block converts each patch (concatenated with its padding mask) into a
    fixed-size embedding vector. The residual/skip connection helps with gradient
    flow during training and allows the network to learn identity mappings easily.
    """

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.hidden_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),  # SiLU is smooth, non-monotonic, helps gradient flow
        )
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        self.residual_layer = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        """Forward pass through the residual block."""
        hidden = self.hidden_layer(x)
        output = self.output_layer(hidden)
        residual = self.residual_layer(x) # we need to project the input to the output dimension
        # Here we learn the residual (the difference) between the input and the output.
        # H(x) = F(x) + x.
        return output + residual

In [ ]:
# Example usage: pass normalized patches to ResidualBlock

# From paper: Along with the input, we also supply a binary padding mask m1:L where 1 denotes
# that the corresponding input in y1:L should be ignored and vice-versa.
patches_cat = torch.cat([norm_patches, pad_mask], dim=-1)  # shape: (num_patches, 2 * patch_len)

# Pass through ResidualBlock to get embeddings
embed_dim = 32
block = ResidualBlock(input_dim= 2 * patch_len, hidden_dim=32, output_dim=embed_dim)
embeddings = block(patches_cat)

print('Original patches shape:', patches_cat.shape)
print('Patch embeddings shape:', embeddings.shape)

In [ ]:
# RMSNorm: Root Mean Square Layer Normalization

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.

    Formula: x_normed = x / sqrt(mean(x^2) + eps) * weight

    Used in TimesFM for pre-attention normalization. The learnable weight
    parameter allows the model to scale the normalized output.
    """

    def __init__(self, dim: int, eps: float = 1e-6, add_unit_offset: bool = False):
        super().__init__()
        self.eps = eps
        self.add_unit_offset = add_unit_offset
        # Initialize weight to zeros (becomes identity with add_unit_offset=True)
        self.weight = nn.Parameter(torch.zeros(dim))

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        """Compute RMS normalization."""
        # rsqrt = 1 / sqrt(x)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply RMS normalization with learnable scaling."""
        # Normalize in float32 for numerical stability
        output = self._norm(x.float())
        if self.add_unit_offset:
            # Scale by (1 + weight), so weight=0 -> identity
            output = output * (1 + self.weight.float())
        else:
            # Scale by weight directly
            output = output * self.weight.float()
        return output.type_as(x)

In [ ]:
# Test: RMSNorm with simple 2D vectors for visualization

# Create simple 2D input vectors (easy to visualize)
simple_input = torch.tensor([
    [2.0, 4.0],    # Vector with RMS = sqrt((4 + 16) / 2) = sqrt(10) ≈ 3.16
    [1.0, 1.0],    # Vector with RMS = sqrt((1 + 1) / 2) = 1.0
    [6.0, 8.0],    # Same direction as [3, 4], but scaled 2x
])
print("Original vectors:")
print(simple_input.numpy())

# Create RMSNorm for 2D vectors
rms_norm_2d = RMSNorm(dim=2, add_unit_offset=True)

# Apply RMSNorm
normalized = rms_norm_2d(simple_input)
print("After RMSNorm:")
print(normalized.detach().numpy())

# Verify: RMS of each row should be ~1.0
rms_per_row = torch.sqrt((normalized ** 2).mean(dim=-1))
print(f"RMS per row: {rms_per_row.detach().numpy()}")

# Visualize: Compare original vs normalized vectors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original vectors
ax1 = axes[0]
colors = ['#e41a1c', '#377eb8', '#4daf4a']
labels = ['[3, 4]', '[1, 1]', '[6, 8]']
for i, (vec, color, label) in enumerate(zip(simple_input, colors, labels)):
    ax1.arrow(0, 0, vec[0].item(), vec[1].item(), head_width=0.2, head_length=0.15,
              fc=color, ec=color, linewidth=2, label=label)
ax1.set_xlim(-1, 9)
ax1.set_ylim(-1, 9)
ax1.set_aspect('equal')
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_title('Original Vectors')
ax1.legend()

# Normalized vectors
ax2 = axes[1]
for i, (vec, color, label) in enumerate(zip(normalized.detach(), colors, labels)):
    ax2.arrow(0, 0, vec[0].item(), vec[1].item(), head_width=0.1, head_length=0.08,
              fc=color, ec=color, linewidth=2, label=label)
# Add unit circle to show normalization
theta = np.linspace(0, 2*np.pi, 100)
ax2.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='Unit circle')
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_title('After RMSNorm')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Rotary Position Embeddings (RoPE)

class RotaryPositionalEmbedding(nn.Module):
    """
    Rotary Position Embeddings (RoPE) for transformer attention.

    RoPE rotates query and key vectors based on their position, encoding
    relative position information directly into the attention mechanism.

    When we compute dot(Q_rotated, K_rotated), the rotation
    angles cancel to produce terms that depend only on (position_q - position_k),
    naturally capturing relative position.
    """

    def __init__(self, d_model: int, max_seq_len: int = 128, base: int = 10000):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.base = base
        # Precompute frequency bands: inv_freq[i] = 1 / base^(2i/d)
        # For d_model=8: tensor([1.0000, 0.1000, 0.0100, 0.0010])
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        # Cache for efficiency (avoids recomputing sin/cos every forward)
        self._seq_len_cached = None
        self._cos_cached = None
        self._sin_cached = None

    def _update_cache(self, seq_len: int, device: torch.device):
        """Update cached cos/sin values if sequence length changes."""
        if seq_len != self._seq_len_cached:
            self._seq_len_cached = seq_len
            # Position indices: [0, 1, 2, ..., seq_len-1]
            t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
            # Compute frequencies for each position: freqs[pos, i] = pos * inv_freq[i]
            freqs = torch.outer(t, self.inv_freq)  # (seq_len, d_model//2)
            # Duplicate for full dimensionality (each pair uses same angle)
            emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, d_model)
            # Cache with batch dimension for broadcasting
            self._cos_cached = emb.cos()[None, :, :]  # (1, seq_len, d_model)
            self._sin_cached = emb.sin()[None, :, :]  # (1, seq_len, d_model)

    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """Rotate half the hidden dims of the input."""
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def apply_rotary_pos_emb(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor
    ) -> torch.Tensor:
        """Apply rotary position embedding to input tensor."""
        return (x * cos) + (self.rotate_half(x) * sin)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply RoPE to input tensor."""
        # Handle both rank-3 [B, T, D] and rank-4 [B, T, num_heads, head_dim]
        if len(x.shape) == 4:
            batch, seq_len, num_heads, head_dim = x.shape
            # Update cache based on flattened dim
            self._update_cache(seq_len, x.device)
            # Get cos/sin and expand for the full dimension
            cos = self._cos_cached[:, :seq_len, :]
            sin = self._sin_cached[:, :seq_len, :]
            # Need to tile cos/sin for each head
            if head_dim == self.d_model:
                # RoPE applies per-head: cos/sin are [1, T, d_model]
                cos = cos.unsqueeze(2)  # [1, T, 1, d_model]
                sin = sin.unsqueeze(2)  # [1, T, 1, d_model]
                return self.apply_rotary_pos_emb(x, cos, sin)
            else:
                raise ValueError(
                    f"For rank-4 input, head_dim ({head_dim}) must match d_model ({self.d_model})"
                )
        elif len(x.shape) == 3:
            _, seq_len, d = x.shape
            if d != self.d_model:
                raise ValueError(f"Input dim {d} must match d_model {self.d_model}")
            self._update_cache(seq_len, x.device)
            return self.apply_rotary_pos_emb(
                x,
                self._cos_cached[:, :seq_len, :],
                self._sin_cached[:, :seq_len, :]
            )
        else:
            raise ValueError(f"Input must be rank 3 or 4, got rank {len(x.shape)}")

In [ ]:
# Test RoPE: Calculate angles for different positions and dimension pairs
d_model = 16
rope = RotaryPositionalEmbedding(d_model=d_model, max_seq_len=128)

# Let's print the inverse frequencies to understand the angle growth
print(f"  {rope.inv_freq.detach().cpu().numpy()}")

# Calculate angles: angle[pos, i] = position × inv_freq[i]
positions = np.arange(10)  # positions 0-9
inv_freq = rope.inv_freq.numpy()

# Create angle matrix: [positions x dimension_pairs]
angles = np.outer(positions, inv_freq)  # shape: (10, d_model//2)
degrees = np.degrees(angles)  # Convert to degrees

print(f"pos=0: {angles[0, :6]}")  # All zeros at position 0
print(f"pos=1 (rad): {angles[1, :4]}")
print(f"pos=1 (deg): {degrees[1, :4]}")
print(f"pos=2 (rad): {angles[2, :4]}")
print(f"pos=2 (deg): {degrees[2, :4]}")
print(f"pos=4 (rad): {angles[4, :4]}")
print(f"pos=4 (deg): {degrees[4, :4]}")

# Plot: Angles by position for each dimension pair
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(len(inv_freq)):
    ax.plot(positions, angles[:, i], 'o-', label=f'Pair {i} (inv_freq={inv_freq[i]:.4f})')
ax.set_xlabel('Position')
ax.set_ylabel('Angle (radians)')
ax.set_title('Angle Growth by Position (Fast dimensions rotate more per step)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# From paper: Each of these layers have the standard multi-head self-attention (SA).

def get_large_negative_number(dtype: torch.dtype) -> torch.Tensor:
    """Return a large negative value suitable for attention masking."""
    if dtype.is_floating_point:
        dtype_max = torch.finfo(dtype).max
    else:
        dtype_max = torch.iinfo(dtype).max
    return torch.tensor(-0.7 * dtype_max, dtype=dtype)


def causal_mask(input_t: torch.Tensor) -> torch.Tensor:
    """
    Compute causal (autoregressive) attention mask for decoder self-attention.

    In autoregressive models, each position can only attend to itself and previous
    positions (not future). This creates a lower-triangular attention pattern.
    The mask is additive: 0 for allowed positions, large negative for blocked.
    """
    t = input_t.shape[1]
    col_idx = torch.arange(t, device=input_t.device).unsqueeze(0).repeat(t, 1)
    row_idx = torch.arange(t, device=input_t.device).unsqueeze(1).repeat(1, t)
    # row_idx < col_idx means "can't attend to future positions"
    mask = (row_idx < col_idx).to(input_t.dtype) * get_large_negative_number(input_t.dtype)
    return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, T, T]


class SelfAttention(nn.Module):
    """Multi-head self-attention with per-dimension scaling, RoPE, and Q/K norm."""

    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        head_dim: int,
        use_rope: bool = True,
        use_qk_norm: bool = True,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.hidden_size = hidden_size
        self.use_rope = use_rope
        self.use_qk_norm = use_qk_norm
        # Per-dimension scaling parameter (learnable)
        self.scaling = nn.Parameter(torch.zeros(head_dim))
        # Combined QKV projection for efficiency (no bias in TimesFM)
        self.qkv_proj = nn.Linear(hidden_size, 3 * num_heads * head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * head_dim, hidden_size, bias=False)
        # Optional RoPE
        if use_rope:
            self.rope = RotaryPositionalEmbedding(d_model=head_dim)
        # Optional Q/K normalization (TimesFM uses RMSNorm)
        if use_qk_norm:
            self.q_norm = RMSNorm(head_dim, add_unit_offset=True)
            self.k_norm = RMSNorm(head_dim, add_unit_offset=True)

    def _per_dim_scaling(self, query: torch.Tensor) -> torch.Tensor:
        """
        Apply learnable per-dimension scaling to queries.

        Unlike standard attention which uses 1/sqrt(d_k) for all dimensions,
        TimesFM learns a separate scaling factor per dimension using softplus.
        """
        r_softplus_0 = 1.442695041  # 1/ln(2)
        scale = r_softplus_0 / math.sqrt(self.head_dim)
        scale = scale * F.softplus(self.scaling)
        return query * scale.view(1, 1, 1, -1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        """Forward pass through self-attention."""
        B, T, _ = x.shape
        # Project to Q, K, V
        qkv = self.qkv_proj(x)
        q_size = k_size = v_size = self.num_heads * self.head_dim
        q, k, v = qkv.split([q_size, k_size, v_size], dim=-1)
        # Reshape: [B, T, num_heads, head_dim]
        q = q.view(B, T, self.num_heads, self.head_dim)
        k = k.view(B, T, self.num_heads, self.head_dim)
        v = v.view(B, T, self.num_heads, self.head_dim)
        # Apply RoPE to Q and K (before normalization)
        if self.use_rope:
            q = self.rope(q)
            k = self.rope(k)
        # Apply Q/K normalization
        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)
        # Transpose for attention: [B, num_heads, T, head_dim]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # Apply per-dimension scaling
        q = self._per_dim_scaling(q)
        # Attention scores: [B, num_heads, T, T]
        scores = torch.matmul(q, k.transpose(-2, -1))
        if mask is not None:
            scores = scores + mask
        attn_weights = F.softmax(scores, dim=-1)
        # Weighted sum of values
        output = torch.matmul(attn_weights, v)
        # Reshape back: [B, T, hidden_size]
        output = output.transpose(1, 2).contiguous().view(B, T, -1)
        output = self.o_proj(output)
        return attn_weights, output

In [ ]:
# Test: Self-Attention on patch embeddings

hidden_size = embed_dim
num_heads = 4
head_dim = hidden_size // num_heads  # 8

# Create self-attention layer
attn = SelfAttention(hidden_size=hidden_size, num_heads=num_heads, head_dim=head_dim)

# Create causal mask for autoregressive decoding
mask = causal_mask(embeddings)

# Run self-attention
attn_weights, attn_output = attn(embeddings, mask=mask)

print(f"Input embeddings shape: {embeddings.shape}")   # [1, 8, 32]
print(f"Attention weights shape: {attn_weights.shape}") # [1, 4, 8, 8] (B, heads, T, T)
print(f"Attention output shape: {attn_output.shape}")  # [1, 8, 32]


# Visualize attention patterns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot causal mask
ax1 = axes[0]
mask_vis = mask[0, 0].detach().cpu().numpy()
mask_vis = np.where(mask_vis < -1e10, -1, 0)
im1 = ax1.imshow(mask_vis, cmap='RdBu', vmin=-1, vmax=0)
ax1.set_title('Causal Mask (blue=allowed, red=blocked)')
ax1.set_xlabel('Key position')
ax1.set_ylabel('Query position')
plt.colorbar(im1, ax=ax1)

# Plot attention weights (first head)
ax2 = axes[1]
attn_vis = attn_weights[0, 0].detach().cpu().numpy()
im2 = ax2.imshow(attn_vis, cmap='viridis')
ax2.set_title('Attention Weights (Head 1) (brighter=more attention)')
ax2.set_xlabel('Key position')
ax2.set_ylabel('Query position')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
# DecoderLayer: TimesFM-style transformer block

class DecoderLayer(nn.Module):
    """TimesFM Decoder Layer with RoPE support. """

    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        num_heads: int,
        head_dim: int = None,
        use_rope: bool = True,
        use_qk_norm: bool = True,
        use_residual: bool = True,
        rms_norm_eps: float = 1e-6,
    ):
        super().__init__()
        if head_dim is None:
            head_dim = hidden_size // num_heads
        self.use_residual = use_residual
        # Pre/post attention normalization
        self.pre_attn_norm = RMSNorm(hidden_size, eps=rms_norm_eps, add_unit_offset=True)
        self.post_attn_norm = RMSNorm(hidden_size, eps=rms_norm_eps, add_unit_offset=True)
        # Self-attention with RoPE and Q/K norm
        self.self_attn = SelfAttention(
            hidden_size=hidden_size,
            num_heads=num_heads,
            head_dim=head_dim,
            use_rope=use_rope,
            use_qk_norm=use_qk_norm,
        )
        # Pre/post FFN normalization
        self.pre_ff_norm = RMSNorm(hidden_size, eps=rms_norm_eps, add_unit_offset=True)
        self.post_ff_norm = RMSNorm(hidden_size, eps=rms_norm_eps, add_unit_offset=True)
        # Feed-forward network
        self.ff_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ff_down = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.activation = nn.ReLU()

    def forward(
        self,
        hidden_states: torch.Tensor,
        mask: torch.Tensor,
        paddings: torch.Tensor = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the decoder layer."""
        # Attention block: pre-norm -> attention -> post-norm -> residual
        residual = hidden_states
        hidden_states = self.pre_attn_norm(hidden_states)
        attn_weights, hidden_states = self.self_attn(hidden_states, mask=mask)
        hidden_states = self.post_attn_norm(hidden_states)
        if self.use_residual:
            hidden_states = residual + hidden_states
        # FFN block: pre-norm -> FFN -> post-norm -> residual
        residual = hidden_states
        hidden_states = self.pre_ff_norm(hidden_states)
        hidden_states = self.ff_up(hidden_states)
        hidden_states = self.activation(hidden_states)
        hidden_states = self.ff_down(hidden_states)
        hidden_states = self.post_ff_norm(hidden_states)
        # Zero out padded positions before adding residual
        if paddings is not None:
            hidden_states = hidden_states * (1.0 - paddings[:, :, None])
        if self.use_residual:
            hidden_states = residual + hidden_states
        return attn_weights, hidden_states

In [ ]:
# Test: Decoder Layer on patch embeddings

hidden_size = embed_dim  # 32 (from ResidualBlock output)
intermediate_size = hidden_size  # TimesFM uses 1x, not 4x
num_heads = 4
head_dim = hidden_size // num_heads  # 8

# Create decoder layer
decoder_layer = DecoderLayer(
    hidden_size=hidden_size,
    intermediate_size=intermediate_size,
    num_heads=num_heads,
    head_dim=head_dim,
)

# Create causal mask for autoregressive decoding
mask = causal_mask(embeddings)

# Create padding mask (same shape as patch sequence)
# paddings[b, t] = 1 means position t in batch b is padding
paddings = torch.zeros(embeddings.shape[0], embeddings.shape[1])

# Run decoder layer
attn_weights, decoder_output = decoder_layer(embeddings, mask=mask, paddings=paddings)

print(f"Input shape {embeddings.shape}")      # [1, 8, 32]
print(f"Attention weights shape: {attn_weights.shape}")    # [1, 4, 8, 8]
print(f"Decoder output shape: {decoder_output.shape}")  # [1, 8, 32]

# Verify residual connections are working (output shouldn't be too different from input initially)
input_norm = embeddings.norm().item()
output_norm = decoder_output.norm().item()
print(f"Input norm: {input_norm:.4f}")
print(f"Output norm: {output_norm:.4f}")
print(f"Ratio: {output_norm/input_norm:.4f} (should be ~1.0 for untrained model)")

In [ ]:
# Stacked Decoder - The full TimesFM decoder backbone

class StackedDecoder(nn.Module):
    """
    TimesFM's stacked decoder architecture.

    This module implements the full decoder backbone:
    1. Tokenizer: ResidualBlock (patch + mask → embedding)
    2. Stacked Transformers: Multiple DecoderLayer with RoPE
    3. Output Projections: ResidualBlock for point/quantile forecasts
    """

    def __init__(
        self,
        input_patch_len: int = 32,
        output_patch_len: int = 128,
        hidden_size: int = 1280,
        num_layers: int = 20,
        num_heads: int = 16,
        intermediate_size: int = None,
        num_quantiles: int = 10,
        use_rope: bool = True,
        use_qk_norm: bool = True,
        use_residual: bool = True,
    ):
        super().__init__()
        self.input_patch_len = input_patch_len
        self.output_patch_len = output_patch_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_quantiles = num_quantiles
        if intermediate_size is None:
            intermediate_size = hidden_size  # TimesFM uses 1x
        # 1. Tokenizer: project (patch + mask) to embeddings
        self.tokenizer = ResidualBlock(
            input_dim=input_patch_len * 2,
            hidden_dim=hidden_size,
            output_dim=hidden_size,
        )
        # 2. Stacked decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(
                hidden_size=hidden_size,
                intermediate_size=intermediate_size,
                num_heads=num_heads,
                use_rope=use_rope,
                use_qk_norm=use_qk_norm,
                use_residual=use_residual,
            )
            for _ in range(num_layers)
        ])
        # 3. Output projections
        self.output_proj_point = ResidualBlock(
            input_dim=hidden_size,
            hidden_dim=hidden_size,
            output_dim=output_patch_len,
        )
        self.output_proj_quantiles = ResidualBlock(
            input_dim=hidden_size,
            hidden_dim=hidden_size,
            output_dim=output_patch_len * num_quantiles,
        )

    def forward(
        self,
        patches: torch.Tensor,
        patch_masks: torch.Tensor,
        paddings: torch.Tensor = None,
    ) -> dict:
        """Forward pass through the stacked decoder."""
        B, N, P = patches.shape
        # Tokenize
        tokenizer_input = torch.cat([patches, patch_masks.to(patches.dtype)], dim=-1)
        input_embeddings = self.tokenizer(tokenizer_input)
        # Causal mask
        mask = causal_mask(input_embeddings)
        # Stacked decoder layers
        hidden_states = input_embeddings
        all_attention_weights = []
        for layer in self.layers:
            attn_weights, hidden_states = layer(hidden_states, mask, paddings)
            all_attention_weights.append(attn_weights)
        output_embeddings = hidden_states
        # Output projections
        point_forecast = self.output_proj_point(output_embeddings)
        quantile_raw = self.output_proj_quantiles(output_embeddings)
        quantile_forecast = quantile_raw.view(B, N, self.output_patch_len, self.num_quantiles)
        return {
            'input_embeddings': input_embeddings,
            'output_embeddings': output_embeddings,
            'point_forecast': point_forecast,
            'quantile_forecast': quantile_forecast,
            'attention_weights': all_attention_weights,
        }

    def count_parameters(self) -> dict:
        """Count parameters by component."""
        def count(m):
            return sum(p.numel() for p in m.parameters())
        return {
            'tokenizer': count(self.tokenizer),
            'decoder_layers': sum(count(l) for l in self.layers),
            'output_proj_point': count(self.output_proj_point),
            'output_proj_quantiles': count(self.output_proj_quantiles),
            'total': count(self),
        }

In [ ]:
# Test: Create a small StackedDecoder and verify shapes

# Use smaller dimensions for testing (actual TimesFM uses much larger)
test_config = {
    'input_patch_len': 8,        # TimesFM: 32
    'output_patch_len': 32,       # TimesFM: 128
    'hidden_size': 64,            # TimesFM: 1280
    'num_layers': 2,              # TimesFM: 20
    'num_heads': 4,               # TimesFM: 16
    'intermediate_size': 64,      # TimesFM: 1280
    'num_quantiles': 10,          # Same as TimesFM
}

# Create model
stacked_decoder = StackedDecoder(**test_config)

# Print parameter counts
param_counts = stacked_decoder.count_parameters()
print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name:25s}: {count:>10,}")

# Create test input (using our existing patches from earlier)
# Reshape patches to match test config
test_patches = patches_batched[:, :4, :8]   # [1, 4, 8] - 4 patches of length 8
test_masks = pad_mask[:, :4, :8]            # [1, 4, 8]
test_paddings = torch.zeros(1, 4)           # [1, 4] - no padding at patch level

print(f"Input patches shape: {test_patches.shape}")
print(f"Input masks shape:   {test_masks.shape}")

# Forward pass
with torch.no_grad():
    outputs = stacked_decoder(test_patches, test_masks, test_paddings)

print(f"Input embeddings:   {outputs['input_embeddings'].shape}")    # [1, 4, 64]
print(f"Output embeddings:  {outputs['output_embeddings'].shape}")   # [1, 4, 64]
print(f"Point forecast:     {outputs['point_forecast'].shape}")      # [1, 4, 32]
print(f"Quantile forecast:  {outputs['quantile_forecast'].shape}")   # [1, 4, 32, 10]
print(f"Attention weights:  {len(outputs['attention_weights'])} layers, each {outputs['attention_weights'][0].shape}")

In [ ]:
# Quantile Loss for Probabilistic Forecasting

def quantile_loss(pred: torch.Tensor, actual: torch.Tensor, quantile: float) -> torch.Tensor:
    """
    Compute quantile (pinball) loss for probabilistic forecasting.

    Quantile loss penalizes:
    - Under-predictions more heavily for high quantiles (e.g., 90th percentile)
    - Over-predictions more heavily for low quantiles (e.g., 10th percentile)

    Example:
        For quantile=0.9:
        - If actual > pred (under-prediction): loss = 2 * 0.9 * error  (high penalty)
        - If actual < pred (over-prediction): loss = 2 * 0.1 * |error| (low penalty)
    """
    if pred.dim() > actual.dim():
        pred = pred.squeeze(-1)
    deviation = actual - pred
    loss_under = deviation * quantile
    loss_over = -deviation * (1.0 - quantile)
    # Take the positive part (whichever applies)
    return 2 * torch.where(loss_under >= 0, loss_under, loss_over)


def compute_loss(
    point_forecast: torch.Tensor,
    quantile_forecast: torch.Tensor,
    target: torch.Tensor,
    quantiles: List[float] = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    point_weight: float = 1.0,
    quantile_weight: float = 1.0,
) -> dict:
    """
    Compute combined training loss for TimesFM.

    Total loss = MSE(point forecast) + sum(quantile losses)
    """
    # Point forecast loss (MSE)
    point_loss = F.mse_loss(point_forecast, target)
    # Quantile losses
    per_quantile_loss = []
    for i, q in enumerate(quantiles):
        q_pred = quantile_forecast[..., i]
        q_loss = quantile_loss(q_pred, target, q).mean()
        per_quantile_loss.append(q_loss)
    total_quantile_loss = torch.stack(per_quantile_loss).mean()
    total_loss = point_weight * point_loss + quantile_weight * total_quantile_loss
    return {
        'total_loss': total_loss,
        'point_loss': point_loss,
        'quantile_loss': total_quantile_loss,
        'per_quantile_loss': per_quantile_loss,
    }

In [ ]:
# Test: Quantile loss behavior visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Generate test data
errors = torch.linspace(-2, 2, 100)
actual = torch.zeros(100)
pred = -errors  # pred = actual - error, so error = actual - pred

# Plot quantile loss for different quantiles
ax1 = axes[0]
for q in [0.1, 0.5, 0.9]:
    losses = quantile_loss(pred, actual, q)
    ax1.plot(errors.numpy(), losses.numpy(), label=f'q={q}')
ax1.set_xlabel('Prediction Error (actual - pred)')
ax1.set_ylabel('Loss')
ax1.set_title('Quantile Loss by Quantile Level')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)

# Compare to MSE
ax2 = axes[1]
mse_loss = errors ** 2
q50_loss = quantile_loss(pred, actual, 0.5)
ax2.plot(errors.numpy(), mse_loss.numpy(), label='MSE', linewidth=2)
ax2.plot(errors.numpy(), q50_loss.numpy(), label='Quantile (q=0.5)', linewidth=2, linestyle='--')
ax2.set_xlabel('Prediction Error')
ax2.set_ylabel('Loss')
ax2.set_title('MSE vs Quantile Loss (q=0.5)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# q=0.9: Penalizes under-predictions heavily. If targeting the 90th percentile, you want predictions to be high enough.
# q=0.1: Penalizes over-predictions heavily. If targeting the 10th percentile, you want predictions to be low enough.
# q=0.5: Median - equal penalty for over/under (equivalent to MAE).

In [ ]:
# TimesFMFromScratch: Complete End-to-End Model

class TimesFMFromScratch(nn.Module):
    """
    Complete TimesFM implementation from scratch.

    This wrapper handles the full forecasting pipeline:
    1. Input preprocessing (padding, patching, normalization)
    2. Transformer encoding via StackedDecoder
    3. Output postprocessing (denormalization, reshape)
    4. Loss computation for training
    """

    # Default quantiles matching TimesFM
    DEFAULT_QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    def __init__(
        self,
        input_patch_len: int = 32,
        output_patch_len: int = 128,
        hidden_size: int = 1280,
        num_layers: int = 20,
        num_heads: int = 16,
        intermediate_size: int = None,
        num_quantiles: int = 10,
        quantiles: List[float] = None,
        pad_val: float = 1123581321.0,
        tolerance: float = 1e-6,
    ):
        super().__init__()

        self.input_patch_len = input_patch_len
        self.output_patch_len = output_patch_len
        self.hidden_size = hidden_size
        self.pad_val = pad_val
        self.tolerance = tolerance
        self.quantiles = quantiles or self.DEFAULT_QUANTILES
        self.num_quantiles = num_quantiles

        if intermediate_size is None:
            intermediate_size = hidden_size

        # Core model
        self.decoder = StackedDecoder(
            input_patch_len=input_patch_len,
            output_patch_len=output_patch_len,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_heads=num_heads,
            intermediate_size=intermediate_size,
            num_quantiles=num_quantiles,
        )

    def _pad_to_patch_multiple(
        self,
        x: torch.Tensor,
        target_len: int = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Pad input to be a multiple of patch length."""
        B, T = x.shape
        # Calculate padding needed
        if target_len is not None:
            pad_len = max(0, target_len - T)
        else:
            remainder = T % self.input_patch_len
            pad_len = (self.input_patch_len - remainder) % self.input_patch_len
        # Pad at the beginning (left-pad) with pad_val
        if pad_len > 0:
            pad_tensor = torch.full((B, pad_len), self.pad_val,
                                   dtype=x.dtype, device=x.device)
            x_padded = torch.cat([pad_tensor, x], dim=1)
            # Padding mask: 1 = padded, 0 = valid
            pad_mask = torch.cat([
                torch.ones(B, pad_len, dtype=x.dtype, device=x.device),
                torch.zeros(B, T, dtype=x.dtype, device=x.device),
            ], dim=1)
        else:
            x_padded = x
            pad_mask = torch.zeros(B, T, dtype=x.dtype, device=x.device)
        return x_padded, pad_mask

    def _reverse_transform(
        self,
        outputs: torch.Tensor,
        stats: Tuple[torch.Tensor, torch.Tensor],
    ) -> torch.Tensor:
        """Denormalize outputs using saved statistics."""
        mu, sigma = stats
        if outputs.dim() == 4:
            # [B, N, H, Q] - quantile outputs
            return outputs * sigma[:, None, None, None] + mu[:, None, None, None]
        else:
            # [B, N, H] - point outputs
            return outputs * sigma[:, None, None] + mu[:, None, None]

    def preprocess(
        self,
        time_series: torch.Tensor,
        context_len: int = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Preprocess raw time series for model input."""
        # Pad to patch multiple
        x_padded, pad_mask = self._pad_to_patch_multiple(time_series, context_len)
        B, T_padded = x_padded.shape
        N = T_padded // self.input_patch_len
        P = self.input_patch_len
        # Reshape to patches
        patches = x_padded.view(B, N, P)
        masks = pad_mask.view(B, N, P)
        norm_patches, stats = _forward_transform(patches, masks, tolerance=self.tolerance)
        # Store stats for denormalization
        self._last_stats = stats
        # Patch-level padding (a patch is padded if ALL values are padded)
        paddings = masks.min(dim=-1)[0]
        return norm_patches, masks, paddings

    def forward(
        self,
        time_series: torch.Tensor,
        context_len: int = None,
    ) -> dict:
        """Forward pass: preprocess -> decode -> postprocess."""
        # Preprocess
        patches, masks, paddings = self.preprocess(time_series, context_len)
        # Decode
        outputs = self.decoder(patches, masks, paddings)
        # Denormalize
        point_denorm = self._reverse_transform(outputs['point_forecast'], self._last_stats)
        quant_denorm = self._reverse_transform(outputs['quantile_forecast'], self._last_stats)
        return {
            'point_forecast': outputs['point_forecast'],
            'quantile_forecast': outputs['quantile_forecast'],
            'point_forecast_denorm': point_denorm,
            'quantile_forecast_denorm': quant_denorm,
            'input_embeddings': outputs['input_embeddings'],
            'output_embeddings': outputs['output_embeddings'],
            'attention_weights': outputs['attention_weights'],
        }

    def forecast(
        self,
        time_series: torch.Tensor,
        horizon: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Generate forecast for a given horizon."""
        outputs = self.forward(time_series)
        # Get last patch output (forecast starts from end of context)
        point = outputs['point_forecast_denorm'][:, -1, :horizon]
        quantiles = outputs['quantile_forecast_denorm'][:, -1, :horizon, :]
        return point, quantiles

    def compute_loss(
        self,
        time_series: torch.Tensor,
        target: torch.Tensor,
    ) -> dict:
        """Compute training loss on a batch. """
        outputs = self.forward(time_series)
        # Get forecast from last patch
        horizon = target.shape[1]
        point = outputs['point_forecast_denorm'][:, -1, :horizon]
        quantiles = outputs['quantile_forecast_denorm'][:, -1, :horizon, :]
        return compute_loss(point, quantiles, target, self.quantiles)

    def count_parameters(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Test: Create and verify TimesFMFromScratch model

# Small config for testing (full TimesFM uses much larger dims)
small_config = {
    'input_patch_len': 8,
    'output_patch_len': 16,
    'hidden_size': 64,
    'num_layers': 2,
    'num_heads': 4,
    'num_quantiles': 10,
}

# Create model
model = TimesFMFromScratch(**small_config)
print(f"Model parameters: {model.count_parameters():,}")

# Test with our tourism data (context from earlier)
test_context = torch.tensor(context, dtype=torch.float32).unsqueeze(0)  # [1, 60]
test_target = torch.tensor(future, dtype=torch.float32).unsqueeze(0)    # [1, 24]

print(f"Context shape: {test_context.shape}")
print(f"Target shape:  {test_target.shape}")

# Forward pass
with torch.no_grad():
    outputs = model(test_context)

print(f"Output shapes:")
print(f"- Point forecast (norm):   {outputs['point_forecast'].shape}")
print(f"- Quantile forecast:       {outputs['quantile_forecast'].shape}")
print(f"- Point forecast (denorm): {outputs['point_forecast_denorm'].shape}")

# Test forecast method
point_pred, quant_pred = model.forecast(test_context, horizon=16)

print(f"Forecast shapes:")
print(f"- Point: {point_pred.shape}")
print(f"- Quantiles: {quant_pred.shape}")

# Test loss computation
loss_dict = model.compute_loss(test_context, test_target[:, :16])

print(f"Loss values (untrained model):")
print(f"- Total loss:    {loss_dict['total_loss'].item():.4f}")
print(f"- Point loss:    {loss_dict['point_loss'].item():.4f}")
print(f"- Quantile loss: {loss_dict['quantile_loss'].item():.4f}")

In [ ]:
# Load Multiple Large Datasets for Training
import gc

def load_gluonts_dataset(name: str) -> List[np.ndarray]:
    """Load a GluonTS dataset and extract as numpy arrays."""
    try:
        ds = get_dataset(name, regenerate=False)
        series = [np.array(entry["target"], dtype=np.float32) for entry in ds.train]
        total_points = sum(len(s) for s in series)
        print(f"  {name}: {len(series)} series, {total_points:,} points")
        return series
    except Exception as e:
        print(f"  {name}: FAILED - {e}")
        return []

# Load multiple datasets - comment out any you don't need
datasets_to_load = [
    "electricity_hourly",  # 321 series, 8.4M points - Energy consumption
    "traffic",             # 862 series, 15M points - Road occupancy
    "m4_daily",            # 4,227 series, 8.9M points - M4 competition
    "m4_weekly",           # 359 series, 1.1M points - M4 competition
]

combined_series = []
for ds_name in datasets_to_load:
    series = load_gluonts_dataset(ds_name)
    combined_series.extend(series[:2000])  # We take only the first 2000 series
    gc.collect()  # Free memory between datasets

# Add the tourism_monthly we already have (the Tourism dataset is small, so increase its weight by repeating it)
for _ in range(4):
    combined_series.extend(all_series)
total_points = sum(len(s) for s in combined_series)

print(f"TOTAL: {len(combined_series):,} series, {total_points:,} data points")
print(f"Average series length: {total_points / len(combined_series):.0f}")

In [ ]:
from torch.utils.data import Dataset, DataLoader

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name()}")
else:
    device = torch.device("cpu")
    print("Using CPU :(")

In [ ]:
# Dataset: PyTorch Dataset for time series forecasting
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
import numpy as np
import torch

class TimeSeriesDataset(Dataset):
    def __init__(self, series_list, context_len=64, horizon_len=32, samples_per_series=10):
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.total_len = context_len + horizon_len
        self.series = []
        self.scalers_mean = []
        self.scalers_scale = []

        # Filter and globally normalize series
        for s in series_list:
            if len(s) >= self.total_len:
                arr = np.ascontiguousarray(s, dtype=np.float32).reshape(-1, 1)
                scaler = StandardScaler()
                arr_normalized = scaler.fit_transform(arr).flatten()
                self.series.append(arr_normalized)
                # Store the stats to reverse the scaling later
                self.scalers_mean.append(np.float32(scaler.mean_[0]))
                self.scalers_scale.append(np.float32(scaler.scale_[0]))

        self.max_starts = [len(s) - self.total_len for s in self.series]
        self.samples_per_series = samples_per_series

    def __len__(self):
        return len(self.series) * self.samples_per_series

    def __getitem__(self, idx):
        series_idx = idx % len(self.series)
        series = self.series[series_idx]
        start = np.random.randint(0, self.max_starts[series_idx] + 1)
        context = series[start:start + self.context_len]
        target = series[start + self.context_len:start + self.total_len]
        # Get the global scaler stats for this specific series
        mean = self.scalers_mean[series_idx]
        scale = self.scalers_scale[series_idx]
        return (
            torch.from_numpy(context.copy()),
            torch.from_numpy(target.copy()),
            torch.tensor(mean),
            torch.tensor(scale)
        )

# Create train/validation split
def train_val_split(series_list, val_ratio=0.1):
    """Split series list into train and validation sets."""
    n_val = max(1, int(len(series_list) * val_ratio))
    # Shuffle indices and split between train and val
    indices = np.random.permutation(len(series_list))
    val_indices = indices[:n_val]
    train_indices = indices[n_val:]
    train_series = [series_list[i] for i in train_indices]
    val_series = [series_list[i] for i in val_indices]
    return train_series, val_series


train_series, val_series = train_val_split(combined_series, val_ratio=0.1)

total_train_points = sum(len(s) for s in train_series)
total_val_points = sum(len(s) for s in val_series)

print(f"Training series: {len(train_series):,} ({total_train_points:,} points)")
print(f"Validation series: {len(val_series):,} ({total_val_points:,} points)")

In [ ]:
# Training Loop with Gradient Accumulation
from tqdm.auto import tqdm
import time

GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = 64 * 4 = 256


def train_one_epoch(model, train_loader, optimizer, device, epoch, accum_steps=4):
    """Train for one epoch with gradient accumulation."""
    model.train()
    total_loss = torch.tensor(0.0, device=device)
    batch_count = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    optimizer.zero_grad(set_to_none=True)
    for batch_idx, (context, target, _, _) in enumerate(pbar):
        context = context.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        outputs = model(context)
        horizon = target.shape[1]
        point = outputs['point_forecast_denorm'][:, -1, :horizon]
        quants = outputs['quantile_forecast_denorm'][:, -1, :horizon, :]
        loss_dict = compute_loss(point, quants, target, model.quantiles)
        loss = loss_dict['total_loss'] / accum_steps
        # Skip batch if loss is NaN/Inf
        if not torch.isfinite(loss):
            print(f"  Skipping batch {batch_idx} - loss is {loss.item()}")
            optimizer.zero_grad(set_to_none=True)
            continue
        loss.backward()
        if (batch_idx + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss = total_loss + loss.detach() * accum_steps
        batch_count += 1
        if batch_count % 50 == 0:
            pbar.set_postfix(loss=f"{(total_loss / batch_count).item():.4f}")
    if batch_count % accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    return {'loss': (total_loss / batch_count).item()}


@torch.inference_mode()
def validate(model, val_loader, device):
    """Validate model."""
    model.eval()
    total_loss = torch.tensor(0.0, device=device)
    total_mae = torch.tensor(0.0, device=device)
    batch_count = 0
    for context, target, mean, scale in val_loader:
        context = context.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        outputs = model(context)
        horizon = target.shape[1]
        point = outputs['point_forecast_denorm'][:, -1, :horizon]
        quants = outputs['quantile_forecast_denorm'][:, -1, :horizon, :]
        loss_dict = compute_loss(point, quants, target, model.quantiles)
        if torch.isfinite(loss_dict['total_loss']) and loss_dict['total_loss'].item() < 100.0:
            point_cpu = point.detach().cpu()
            target_cpu = target.detach().cpu()
            mean_cpu = mean.detach().cpu().view(-1, 1)
            scale_cpu = scale.detach().cpu().view(-1, 1)
            raw_point = (point_cpu * scale_cpu) + mean_cpu
            raw_target = (target_cpu * scale_cpu) + mean_cpu
            batch_mae = torch.nn.functional.l1_loss(raw_point, raw_target)
            if torch.isfinite(batch_mae):
                total_loss = total_loss + loss_dict['total_loss']
                total_mae = total_mae + batch_mae
                batch_count += 1
    return {
        'loss': (total_loss / batch_count).item(),
        'mae': (total_mae / batch_count).item()
    }


def train_model(
    model,
    train_loader,
    val_loader,
    epochs=20,
    lr=1e-3,
    weight_decay=1e-4,
    device='cpu',
    accum_steps=4,
    batch_size=64,
):
    """Full training loop with gradient accumulation."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'lr': []}
    best_val_loss = float('inf')
    best_state = None
    effective_batch = batch_size * accum_steps
    print(f"Training on {device} for {epochs} epochs")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Batch size: {batch_size} x {accum_steps} accum = {effective_batch} effective")
    start_time = time.time()
    for epoch in range(epochs):
        epoch_start = time.time()
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, device, epoch, accum_steps
        )
        val_metrics = validate(model, val_loader, device)
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_mae'].append(val_metrics['mae'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        scheduler.step()
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train: {train_metrics['loss']:.4f} | "
              f"Val: {val_metrics['loss']:.4f} | "
              f"MAE: {val_metrics['mae']:.4f} | "
              f"{epoch_time:.1f}s/epoch | "
              f"Total: {total_time/60:.1f}min")
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

In [ ]:
# Prepare for training

train_series_large, val_series_large = train_val_split(combined_series, val_ratio=0.1)

MODEL_CONFIG = {
    'input_patch_len': 32,
    'output_patch_len': 64,
    'hidden_size': 64,
    'num_layers': 4,
    'num_heads': 4,
    'num_quantiles': 10,
}

TRAIN_CONFIG = {
    'context_len': 128,
    'horizon_len': 64,
    'batch_size': 64,
    'epochs': 90,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'samples_per_series': 50,
}

train_dataset = TimeSeriesDataset(
    train_series_large,
    context_len=TRAIN_CONFIG['context_len'],
    horizon_len=TRAIN_CONFIG['horizon_len'],
    samples_per_series=TRAIN_CONFIG['samples_per_series'],
)
val_dataset = TimeSeriesDataset(
    val_series_large,
    context_len=TRAIN_CONFIG['context_len'],
    horizon_len=TRAIN_CONFIG['horizon_len'],
    samples_per_series=10,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
)

model = TimesFMFromScratch(**MODEL_CONFIG)

print(f"Parameters: {model.count_parameters():,}")
# From the above:
# Training series: 4,506 (23,209,804 points)
# Validation series: 500 (2,663,381 points)

# We have 4,075 valid training series.
# 4,075 series × 50 samples = 203,750 total training sequences.
# 203,750 / 64 ~ 3,184
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Start Training!

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=TRAIN_CONFIG['epochs'],
    lr=TRAIN_CONFIG['lr'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
    device=device,
    accum_steps=GRADIENT_ACCUMULATION_STEPS,
)
# Total Data Points: 25,873,185
# 25,873,185 / 32 = 808,537 Tokens.
# 808,537 / 211,840 = 3.81
# We have 3.8 tokens per parameter.

In [ ]:
import torch
import timesfm

# Side-by-Side Comparison: TimesFM vs Our Model on Tourism Data

def plot_forecast(ax, context, future, point, quantiles, label, color, title):
    """Plot a single forecast with context, ground truth, and prediction."""
    x_ctx = np.arange(len(context))
    x_fut = np.arange(len(context), len(context) + len(future))

    ax.plot(x_ctx, context, label="Context", color="steelblue")
    ax.plot(x_fut, future, label="Ground Truth", color="black", linestyle=":", linewidth=2)
    ax.plot(x_fut, point[:len(future)], label=label, color=color, linewidth=2)
    ax.fill_between(x_fut, quantiles[:len(future), 0], quantiles[:len(future), 8],
                    color=color, alpha=0.2, label="80% CI")
    ax.axvline(len(context)-0.5, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Time (months)")
    ax.set_ylabel("Value")
    ax.set_title(title)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

# Prepare test data
test_series = next(s for s in all_series if len(s) > 150)
context_len, pred_len = 60, 24
test_context = test_series[-(context_len + pred_len):-pred_len]
test_future = test_series[-pred_len:]
print(f"Test context: {test_context.shape}, Future: {test_future.shape}")

# Load the PyTorch-based TimesFM 2.5 model directly (using the new API)
tfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

# Prepare input as a tensor [Batch, Context_Length]
forecast_input_tensor = torch.tensor([test_context.astype('float32')], dtype=torch.float32)

# In TimesFM 2.5, use ForecastConfig to define inference constraints
tfm_config = timesfm.ForecastConfig(
    max_context=context_len, # Use the actual context length for this test case
    max_horizon=pred_len,
    normalize_inputs=True
)

# Compile the model with the forecast configuration
tfm_model.compile(forecast_config=tfm_config)

# Perform forecast with the required 'inputs' and 'horizon' arguments
tfm_point_raw, tfm_quant_raw = tfm_model.forecast(
    inputs=forecast_input_tensor,
    horizon=pred_len
)

# TimesFM 2.5 returns numpy arrays directly from the forecast method.
# Extract the single series result from the batch.
tfm_point, tfm_quant = tfm_point_raw[0], tfm_quant_raw[0]

# 3. Run Our Model
model.eval()
with torch.inference_mode():
    our_point, our_quant = model.forecast(
        torch.tensor(test_context, dtype=torch.float32).unsqueeze(0).to(device),
        horizon=pred_len
    )
print(f"Model Raw Output Sample: {our_point[0, :3]}")
print(f"Ground Truth Sample: {test_future[:3]}")
our_point, our_quant = our_point.cpu().numpy()[0], our_quant.cpu().numpy()[0]


# 4. Plot side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
plot_forecast(ax1, test_context, test_future, tfm_point, tfm_quant,
              "TimesFM", "crimson", "TimesFM (Pre-trained 200M params)")
plot_forecast(ax2, test_context, test_future, our_point, our_quant,
              "Our Model", "green", f"Our Model ({model.count_parameters():,} params)")

# Sync y-axes for fair comparison
y_min = min(ax1.get_ylim()[0], ax2.get_ylim()[0])
y_max = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
ax1.set_ylim(y_min, y_max)
ax2.set_ylim(y_min, y_max)

plt.suptitle("Tourism Monthly Forecast Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 5. Print metrics comparison
tfm_metrics = compute_metrics(tfm_point[:pred_len], test_future)
our_metrics = compute_metrics(our_point[:pred_len], test_future)

print(f"{'Metric':<10} {'TimesFM':>15} {'Our Model':>15} {'Diff':>15}")
for metric in ['MAE', 'RMSE', 'MAPE']:
    tfm_val, our_val = tfm_metrics[metric], our_metrics[metric]
    diff = our_val - tfm_val
    diff_pct = (diff / tfm_val) * 100 if tfm_val != 0 else 0
    unit = '%' if metric == 'MAPE' else ''
    print(f"{metric:<10} {tfm_val:>14.2f}{unit} {our_val:>14.2f}{unit} {diff:>+14.2f} ({diff_pct:+.1f}%)")